# Stage 2 / 04 — Prepare CLRKD curve dataset

This notebook builds the BDD100K curve-lane dataset used by CLRKDNet and by the in-house fusion trainer. It is Colab-safe: raw archives are read from Drive, work happens in `/content`, and the final dataset is packed back to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q yacs tqdm pyyaml opencv-python-headless tensorboard

In [ ]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
DRIVE_ECOCAR = '/content/drive/MyDrive/EcoCAR'
DRIVE_DATASETS = os.path.join(DRIVE_ECOCAR, 'datasets')
PROJECT_LOCAL = REPO_ROOT
os.makedirs(DRIVE_DATASETS, exist_ok=True)
os.chdir(PROJECT_LOCAL)
if PROJECT_LOCAL not in sys.path:
    sys.path.insert(0, PROJECT_LOCAL)
print('repo:', PROJECT_LOCAL)
print('dataset output:', os.path.join(DRIVE_DATASETS, 'bdd100k_clrkd_curve.tar'))

## 1. Extract raw BDD100K resources to `/content`

In [ ]:
!python stage2/scripts/00_prepare_rmt_dataset_links.py \
    --dataset-root /content/bdd100k_vehicle5 \
    --raw-root /content/bdd100k_raw \
    --downloads-root /content/drive/MyDrive/EcoCAR/downloads

## 2. Generate curve labels and pack the complete dataset tar

The archive must contain `images/`, per-image `.lines.txt` files, `list/`, `masks/`, and `prepare_summary.json`.

In [ ]:
OUT_TAR = os.path.join(DRIVE_DATASETS, 'bdd100k_clrkd_curve.tar')
!python stage2/scripts/04_prepare_bdd_curve_labels.py \
    --dataset-root /content/bdd100k_vehicle5 \
    --raw-root /content/bdd100k_raw \
    --downloads-root /content/drive/MyDrive/EcoCAR/downloads \
    --auto-extract \
    --output-root /content/bdd100k_clrkd_curve \
    --pack-to {OUT_TAR}

## 3. Loud preflight

In [ ]:
import json, os, subprocess
summary_path = '/content/bdd100k_clrkd_curve/prepare_summary.json'
assert os.path.exists(summary_path), summary_path
summary = json.load(open(summary_path))
print(json.dumps(summary, indent=2))
for required in ['images/train', 'images/val', 'list/train_gt.txt', 'list/val.txt', 'masks/train', 'masks/val']:
    path = os.path.join('/content/bdd100k_clrkd_curve', required)
    assert os.path.exists(path), path
print('tar contents preview:')
subprocess.check_call(['tar', '-tf', OUT_TAR], stdout=open('/content/tar_list.txt', 'w'))
print(open('/content/tar_list.txt').read().splitlines()[:20])

## 4. Visual sanity check

In [ ]:
import os, random, cv2, matplotlib.pyplot as plt
root = '/content/bdd100k_clrkd_curve'
items = [x.split()[0].lstrip('/') for x in open(os.path.join(root, 'list/train_gt.txt')).read().splitlines() if x.strip()]
items = items[:200]
for rel in random.sample(items, min(3, len(items))):
    img_path = os.path.join(root, rel)
    line_path = os.path.splitext(img_path)[0] + '.lines.txt'
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 5))
    plt.imshow(img)
    if os.path.exists(line_path):
        for row in open(line_path).read().splitlines():
            vals = [float(v) for v in row.split()]
            xs = vals[0::2]
            ys = vals[1::2]
            plt.plot(xs, ys, linewidth=2)
    plt.axis('off')
    plt.show()